In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
.appName("Spark Dirty Data Clean") \
.master("local[2]") \
.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/30 15:22:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [34]:
! curl -o /opt/examples/datasets/dirty_store_transactions.csv \
https://raw.githubusercontent.com/erkansirin78/datasets/master/dirty_store_transactions.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2548k  100 2548k    0     0  2738k      0 --:--:-- --:--:-- --:--:-- 2740k


In [35]:
df = spark.read.option("header", True).csv("file:/opt/examples/datasets/dirty_store_transactions.csv")

In [60]:
ls -l

total 2576
-rw-r--r--. 1 root root 2609524 Oct 30 15:41 dirty_store_transactions.csv
-rw-r--r--. 1 root root   23735 Oct 30 16:03 spark_dirty_data_clean-oguzhan-metinoglu.ipynb


In [39]:
df.printSchema()

root
 |-- STORE_ID: string (nullable = true)
 |-- STORE_LOCATION: string (nullable = true)
 |-- PRODUCT_CATEGORY: string (nullable = true)
 |-- PRODUCT_ID: string (nullable = true)
 |-- MRP: string (nullable = true)
 |-- CP: string (nullable = true)
 |-- DISCOUNT: string (nullable = true)
 |-- SP: string (nullable = true)
 |-- Date: string (nullable = true)



In [43]:
from pyspark.sql import functions as F

string_cols = ["STORE_ID", "STORE_LOCATION", "PRODUCT_CATEGORY"]

num_cols = ["PRODUCT_ID", "MRP", "CP", "DISCOUNT", "SP"]

for c in string_cols:
    df = df.withColumn(c, F.trim(F.regexp_replace(F.col(c), r'[^a-zA-Z0-9\s]', '')))

for c in num_cols:
    df = df.withColumn(c, F.regexp_replace(F.col(c), r'[^0-9.]', ''))

In [45]:
df.printSchema()
df.show(10, truncate=False)

root
 |-- STORE_ID: string (nullable = true)
 |-- STORE_LOCATION: string (nullable = true)
 |-- PRODUCT_CATEGORY: string (nullable = true)
 |-- PRODUCT_ID: string (nullable = true)
 |-- MRP: string (nullable = true)
 |-- CP: string (nullable = true)
 |-- DISCOUNT: string (nullable = true)
 |-- SP: string (nullable = true)
 |-- Date: string (nullable = true)

+--------+--------------+----------------+----------+---+-----+--------+-----+----------+
|STORE_ID|STORE_LOCATION|PRODUCT_CATEGORY|PRODUCT_ID|MRP|CP   |DISCOUNT|SP   |Date      |
+--------+--------------+----------------+----------+---+-----+--------+-----+----------+
|YR7220  |New York      |Electronics     |12254943  |31 |20.77|1.86    |29.14|2019-11-26|
|YR7220  |New York      |Furniture       |72619323  |15 |9.75 |1.5     |13.5 |2019-11-26|
|YR7220  |New York      |Electronics     |34161682  |88 |62.48|4.4     |83.6 |2019-11-26|
|YR7220  |New York      |Kitchen         |79411621  |91 |58.24|3.64    |87.36|2019-11-26|
|YR7220  

In [47]:
from pyspark.sql.types import IntegerType, FloatType

df = (df
    .withColumn("PRODUCT_ID", F.col("PRODUCT_ID").cast(IntegerType()))
    .withColumn("MRP", F.col("MRP").cast(FloatType()))
    .withColumn("CP", F.col("CP").cast(FloatType()))
    .withColumn("DISCOUNT", F.col("DISCOUNT").cast(FloatType()))
    .withColumn("SP", F.col("SP").cast(FloatType()))
    .withColumn("Date_Casted", F.to_date(F.col("Date"), "yyyy-MM-dd"))
    .drop("Date")
)


In [56]:
df.printSchema()
df.show(10, truncate=False)

root
 |-- STORE_ID: string (nullable = true)
 |-- STORE_LOCATION: string (nullable = true)
 |-- PRODUCT_CATEGORY: string (nullable = true)
 |-- PRODUCT_ID: integer (nullable = true)
 |-- MRP: float (nullable = true)
 |-- CP: float (nullable = true)
 |-- DISCOUNT: float (nullable = true)
 |-- SP: float (nullable = true)
 |-- Date_Casted: date (nullable = true)

+--------+--------------+----------------+----------+----+-----+--------+-----+-----------+
|STORE_ID|STORE_LOCATION|PRODUCT_CATEGORY|PRODUCT_ID|MRP |CP   |DISCOUNT|SP   |Date_Casted|
+--------+--------------+----------------+----------+----+-----+--------+-----+-----------+
|YR7220  |New York      |Electronics     |12254943  |31.0|20.77|1.86    |29.14|2019-11-26 |
|YR7220  |New York      |Furniture       |72619323  |15.0|9.75 |1.5     |13.5 |2019-11-26 |
|YR7220  |New York      |Electronics     |34161682  |88.0|62.48|4.4     |83.6 |2019-11-26 |
|YR7220  |New York      |Kitchen         |79411621  |91.0|58.24|3.64    |87.36|2019-1

In [59]:
df.printSchema()
df.limit(10).toPandas()

root
 |-- STORE_ID: string (nullable = true)
 |-- STORE_LOCATION: string (nullable = true)
 |-- PRODUCT_CATEGORY: string (nullable = true)
 |-- PRODUCT_ID: integer (nullable = true)
 |-- MRP: float (nullable = true)
 |-- CP: float (nullable = true)
 |-- DISCOUNT: float (nullable = true)
 |-- SP: float (nullable = true)
 |-- Date_Casted: date (nullable = true)



,STORE_ID,STORE_LOCATION,PRODUCT_CATEGORY,PRODUCT_ID,MRP,CP,DISCOUNT,SP,Date_Casted
0,YR7220,New York,Electronics,12254943,31.0,20.770000,1.86,29.139999,2019-11-26
1,YR7220,New York,Furniture,72619323,15.0,9.750000,1.50,13.500000,2019-11-26
2,YR7220,New York,Electronics,34161682,88.0,62.480000,4.40,83.599998,2019-11-26
3,YR7220,New York,Kitchen,79411621,91.0,58.240002,3.64,87.360001,2019-11-26
4,YR7220,New York,Fashion,39520263,85.0,51.000000,2.55,82.449997,2019-11-26
5,YR7220,New York,Kitchen,93809204,37.0,24.049999,0.74,36.259998,2019-11-26
6,YR7220,New York,Cosmetics,86610412,80.0,48.799999,6.40,73.599998,2019-11-26
7,YR7220,New York,Kitchen,52503356,71.0,42.599998,5.68,65.320000,2019-11-26
8,YR7220,New York,Kitchen,77516479,92.0,56.119999,3.68,88.320000,2019-11-26
9,YR7220,New York,Cosmetics,47334289,16.0,10.720000,0.96,15.040000,2019-11-26
